# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

In [2]:
%pip install evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b4cd8493298317e1778606d376555cb8c6c864e1dff9376d8f4f8498c07ac35c
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [3]:
import torch
SEED = 101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [4]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

data/all/train-00000-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00001-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00002-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

...

data/all/train-00007-of-00009.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

data/all/train-00008-of-00009.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

data/all/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/54948 [00:00<?, ? examples/s]

In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [5]:
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

Filter:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Filter:   0%|          | 0/54948 [00:00<?, ? examples/s]

In [6]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [7]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

In [8]:
def format_input_output(example):

    messages = example["messages"]

    prompt_parts = []
    response = None

    for msg in messages:

        role = msg["role"]
        content = msg["content"]

        if role == "system":
            prompt_parts.append(f"System: {content}")

        elif role == "user":
            prompt_parts.append(f"User: {content}")

        elif role == "assistant":
            response = content

    prompt_parts.append("Assistant:")

    prompt = "\n".join(prompt_parts)

    return {
        "prompt": prompt,
        "response": response
    }

Apply the function you implemented to the dataset as a whole.

In [9]:
ds_sft = smoltalk_simplified.map(format_input_output)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Then verify that the dataset now contains the new fields you created.

In [10]:
ds_sft['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "System: You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.\nUser: Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.\nAssistant:",
 'response': 'The chef made more food after the restaurant ran out.'}

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

In [16]:
print("pad:", tokenizer.pad_token)
print("eos:", tokenizer.eos_token)

pad: <|endoftext|>
eos: <|endoftext|>


Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [17]:
def tokenize_helper(example):

    prompt = example["prompt"]
    response = example["response"]

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + response_ids

    labels = [-100] * len(prompt_ids) + response_ids

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [18]:
tokenized_ds_sft = ds_sft.map(tokenize_helper)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [19]:
tokenized_ds_sft["train"][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "System: You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.\nUser: Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.\nAssistant:",
 'response': 'The chef made more food after the restaurant ran out.',
 'input_ids': [18403,
  42,
  1206,
  359,
  354,
  5646,
  298,
  12021,
  11173,
  30,
  1206,
  523,
  325,
  2711,
  351,
  253,
  1694,
  284,
  346,
  737,
  2

As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [20]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [21]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(
                pred_ids_filtered, skip_special_tokens=True,
                eos_token_id=self.tokenizer.vocab['<|im_end|>']
            )

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}

compute_metrics = RougeMetricComputer(tokenizer)


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [22]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

In [24]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import time
import json

print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL")
print("=" * 80)

pretrained_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
).to("cuda")

pretrained_eval_args = TrainingArguments(
    eval_strategy="no",
    per_device_eval_batch_size=1,
    bf16=False,
    fp16=True,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

pretrained_trainer = make_trainer(
    pretrained_model,
    pretrained_eval_args
)

t0 = time.perf_counter()

pretrained_eval_metrics = pretrained_trainer.evaluate()

pretrained_eval_time = time.perf_counter() - t0

pretrained_eval_loss = float(
    pretrained_eval_metrics["eval_loss"]
)

pretrained_rougeL = pretrained_eval_metrics.get(
    "eval_rougeL",
    None
)

print("\nPRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))

print("\nEvaluation time:")
print(pretrained_eval_time)


EVALUATING PRETRAINED MODEL


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

  0%|          | 0/400 [00:00<?, ?it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 2.1280581951141357,
  "eval_model_preparation_time": 0.0041,
  "eval_rougeL": 0.5978924681160525,
  "eval_runtime": 41.9781,
  "eval_samples_per_second": 9.529,
  "eval_steps_per_second": 9.529
}

Evaluation time:
42.00901625300003



## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

In [26]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import torch
import time
import json

baseline_training_args = TrainingArguments(
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=False,
    fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to("cuda")

baseline_trainer = make_trainer(
    base_model,
    baseline_training_args
)

print("\n" + "=" * 80)
print("TRAINING FULL MODEL")
print("=" * 80)

t0 = time.perf_counter()

baseline_trainer.train()

training_time = time.perf_counter() - t0

baseline_metrics = baseline_trainer.evaluate()

print("\nFULL FINETUNING RESULTS:")
print(json.dumps(baseline_metrics, indent=2))

print("\nTraining time:")
print(training_time)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


TRAINING FULL MODEL


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.384', 'grad_norm': '1.632', 'learning_rate': '3.001e-05', 'epoch': '0.4'}
{'loss': '1.201', 'grad_norm': '5.91', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.192', 'eval_rougeL': '0.663', 'eval_runtime': '32.12', 'eval_samples_per_second': '12.45', 'eval_steps_per_second': '12.45', 'epoch': '1'}
{'train_runtime': '783.8', 'train_samples_per_second': '6.379', 'train_steps_per_second': '6.379', 'train_loss': '1.262', 'epoch': '1'}


  0%|          | 0/400 [00:00<?, ?it/s]


FULL FINETUNING RESULTS:
{
  "eval_loss": 1.1918531656265259,
  "eval_rougeL": 0.6630056975605902,
  "eval_runtime": 31.1012,
  "eval_samples_per_second": 12.861,
  "eval_steps_per_second": 12.861,
  "epoch": 1.0
}

Training time:
784.4574969260002


### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [27]:
def num_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

In [28]:
print(num_trainable_parameters(base_model))

134515008


Apply this function to the SFT-trained model and check that the result makes sense.

## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [29]:
def extract_lora_targets(model):

    targets = {}

    for name, module in model.named_modules():

        if any(
            x in name
            for x in ["q_proj", "k_proj", "v_proj", "o_proj"]
        ):
            targets[name] = module

    return targets

In [30]:
targets = extract_lora_targets(base_model)

print(len(targets))

list(targets.keys())[:10]

120


['model.layers.0.self_attn.q_proj',
 'model.layers.0.self_attn.k_proj',
 'model.layers.0.self_attn.v_proj',
 'model.layers.0.self_attn.o_proj',
 'model.layers.1.self_attn.q_proj',
 'model.layers.1.self_attn.k_proj',
 'model.layers.1.self_attn.v_proj',
 'model.layers.1.self_attn.o_proj',
 'model.layers.2.self_attn.q_proj',
 'model.layers.2.self_attn.k_proj']

We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [31]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [32]:
import torch
import torch.nn as nn

class LoRALayer(nn.Module):

    def __init__(self, W, r, alpha):
        super().__init__()

        self.W = W

        for p in self.W.parameters():
            p.requires_grad = False

        self.r = r
        self.alpha = alpha

        in_features = W.in_features
        out_features = W.out_features

        self.A = nn.Parameter(
            torch.randn(r, in_features) * 0.01
        )

        self.B = nn.Parameter(
            torch.zeros(out_features, r)
        )

    def forward(self, x):

        original = self.W(x)

        lora = (x @ self.A.T) @ self.B.T

        return original + (self.alpha / self.r) * lora

In [33]:
test_layer = list(targets.values())[0]

lora_layer = LoRALayer(
    test_layer,
    r=8,
    alpha=16
)

print(lora_layer)

LoRALayer(
  (W): Linear(in_features=576, out_features=576, bias=False)
)


Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.


In [36]:
from transformers import AutoModelForCausalLM
from transformers import TrainingArguments
import torch
import time
import json

# =========================
# Load model
# =========================

lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32
).to("cuda")

# =========================
# Freeze ALL parameters
# =========================

for p in lora_model.parameters():
    p.requires_grad = False

# =========================
# Replace attention layers with LoRA
# =========================

targets = extract_lora_targets(lora_model)

lora_layers = {}

for name, layer in targets.items():
    lora_layers[name] = LoRALayer(
        layer,
        r=8,
        alpha=16
    )

lora_model = replace_layers(
    lora_model,
    lora_layers
)

# =========================
# Enable ONLY LoRA params
# =========================

for module in lora_model.modules():
    if isinstance(module, LoRALayer):
        module.A.requires_grad = True
        module.B.requires_grad = True

print("LoRA target layers:", len(lora_layers))
print("Trainable parameters:", num_trainable_parameters(lora_model))

# =========================
# Training setup
# =========================

lora_training_args = TrainingArguments(
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=False,
    fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

lora_trainer = make_trainer(
    lora_model,
    lora_training_args
)

print("\n" + "=" * 80)
print("TRAINING TRUE LORA MODEL")
print("=" * 80)

t0 = time.perf_counter()

lora_trainer.train()

lora_training_time = time.perf_counter() - t0

lora_metrics = lora_trainer.evaluate()

print("\nTRUE LORA RESULTS:")
print(json.dumps(lora_metrics, indent=2))

print("\nLoRA training time:")
print(lora_training_time)

print("\nFinal trainable parameters:")
print(num_trainable_parameters(lora_model))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

LoRA target layers: 120
Trainable parameters: 921600

TRAINING TRUE LORA MODEL


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.559', 'grad_norm': '0.8718', 'learning_rate': '3.001e-05', 'epoch': '0.4'}
{'loss': '1.366', 'grad_norm': '1.845', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.421', 'eval_rougeL': '0.6347', 'eval_runtime': '38.27', 'eval_samples_per_second': '10.45', 'eval_steps_per_second': '10.45', 'epoch': '1'}
{'train_runtime': '877', 'train_samples_per_second': '5.701', 'train_steps_per_second': '5.701', 'train_loss': '1.436', 'epoch': '1'}


  0%|          | 0/400 [00:00<?, ?it/s]


TRUE LORA RESULTS:
{
  "eval_loss": 1.4212826490402222,
  "eval_rougeL": 0.634680963819511,
  "eval_runtime": 38.5084,
  "eval_samples_per_second": 10.387,
  "eval_steps_per_second": 10.387,
  "epoch": 1.0
}

LoRA training time:
877.4818974899999

Final trainable parameters:
921600


In [37]:
print(num_trainable_parameters(lora_model))

921600


### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

In [ ]:
The pretrained model was able to generate coherent text,
but it did not always follow instructions accurately.

The fully fine-tuned model produced responses that were
more relevant to the user instructions and generally
followed the requested format.

The LoRA model also demonstrated instruction-following
behavior and produced outputs similar to the fully
fine-tuned model, although the responses were sometimes
slightly less accurate or detailed.

Overall, both fine-tuned models learned instruction-following
behavior, while the LoRA model achieved this using
significantly fewer trainable parameters.